# EV Battery Health Prediction Model
**Course**: Machine Learning Project / Lab  
**Objective**: Predict Electric Vehicle (EV) battery health (%) based on driving history, usage patterns, and thermal conditions.

### Step 1: Import Required Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor,
    VotingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("ev_battery_health_data.csv")
df.head()

### Step 2: Data Exploration & Heatmap

In [ ]:
print('Dataset Shape:', df.shape)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Summary Statistics ---')
df.describe()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Features')
plt.show()

### Step 3: Feature Matrix (X) and Target Vector (y) Split

In [ ]:
X = df.drop(columns=['battery_health_%'])
y = df['battery_health_%']

# Split into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

### Step 4: Model Training (Individual Baseline Models & 3 Hybrid Models)
We train baseline models and 3 distinct hybrid models:
1. **Hybrid 1 (Stacking Regressor)**: Meta-learner (Ridge) combining Linear, RF, and GB models.
2. **Hybrid 2 (Voting / Blending Regressor)**: Weighted consensus averaging of diverse model predictions.
3. **Hybrid 3 (Two-Stage Residual Hybrid)**: Linear baseline regression + Gradient Boosting on residual errors.

In [ ]:
# Linear Regression Model
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print('--- Linear Regression ---')
print('MAE :', mean_absolute_error(y_test, pred_lr))
print('RMSE:', np.sqrt(mean_squared_error(y_test, pred_lr)))
print('R2  :', r2_score(y_test, pred_lr))

In [ ]:
# Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print('--- Random Forest Regressor ---')
print('MAE :', mean_absolute_error(y_test, pred_rf))
print('RMSE:', np.sqrt(mean_squared_error(y_test, pred_rf)))
print('R2  :', r2_score(y_test, pred_rf))

#### Implementing the 3 Hybrid Models

In [ ]:
# Define Two-Stage Residual Hybrid Regressor
class TwoStageResidualHybrid(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators=80, learning_rate=0.05, max_depth=3, random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state
        self.stage1_linear = LinearRegression()
        self.stage2_booster = GradientBoostingRegressor(
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            random_state=self.random_state
        )

    def fit(self, X, y):
        self.stage1_linear.fit(X, y)
        residuals = y - self.stage1_linear.predict(X)
        self.stage2_booster.fit(X, residuals)
        return self

    def predict(self, X):
        return self.stage1_linear.predict(X) + self.stage2_booster.predict(X)

# Base estimators
base_lr = LinearRegression()
base_rf = RandomForestRegressor(n_estimators=120, max_depth=10, min_samples_leaf=2, random_state=42, n_jobs=-1)
base_gb = GradientBoostingRegressor(n_estimators=80, learning_rate=0.05, max_depth=3, random_state=42)

# 1. Stacking Hybrid
hybrid_stacking = StackingRegressor(
    estimators=[("lr", base_lr), ("rf", base_rf), ("gb", base_gb)],
    final_estimator=Ridge(),
    n_jobs=-1
)

# 2. Voting / Blending Hybrid
hybrid_voting = VotingRegressor(
    estimators=[("lr", base_lr), ("rf", base_rf), ("gb", base_gb)],
    n_jobs=-1
)

# 3. Two-Stage Residual Hybrid
hybrid_residual = TwoStageResidualHybrid()

# Train and evaluate all 3 hybrids
hybrid_models = {
    "Hybrid 1: Stacking Regressor": hybrid_stacking,
    "Hybrid 2: Voting Regressor": hybrid_voting,
    "Hybrid 3: Two-Stage Residual": hybrid_residual
}

for name, model in hybrid_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name}:")
    print(f"  MAE : {mean_absolute_error(y_test, pred):.3f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.3f}")
    print(f"  R2  : {r2_score(y_test, pred):.4f}
")


### Step 5: Visualizing Results

In [ ]:
# Scatter plot: Actual vs Predicted
plt.figure(figsize=(8, 5))
plt.scatter(y_test, pred_rf, alpha=0.4, color='royalblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Battery Health (%)')
plt.ylabel('Predicted Battery Health (%)')
plt.title('Actual vs Predicted Battery Health (Random Forest)')
plt.grid(True)
plt.show()

In [ ]:
# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='darkcyan')
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.show()

### Step 6: Sample Custom Prediction

In [ ]:
# Test prediction on custom EV input data
sample_input = pd.DataFrame([{
    'total_distance_km': 15000,
    'average_trip_speed_kmph': 40,
    'ambient_temperature_C': 30,
    'trip_duration_min': 50,
    'charging_cycles': 250,
    'fast_charging_ratio_%': 20,
    'average_battery_temperature_C': 35
}])

predicted_health = rf.predict(sample_input)[0]
print(f'Predicted Battery Health: {predicted_health:.2f}%')